#**Predicting Chess Games using FEN notation @ diferring intervals (outcome and winner):**

*Install missing requirements*

In [2]:
%pip install scipy
%pip install tensorflow
%pip install pandas
%pip install seaborn
%pip install numpy
%pip install scikit-learn
%pip install matplotlib
%pip install python-chess

#### Imports

In [3]:
#Imports

import pandas as pd
import seaborn
import numpy as np
import matplotlib.pyplot as plt
import chess
import random
import re

#imports for transformed model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, train_test_split, cross_val_score
from sklearn.utils.class_weight import compute_class_weight


#tensorflow imports for transformer model
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.layers import TextVectorization, Embedding, Dense, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D, Dropout, Concatenate

#### Dataset import and feature extraction

In [4]:
#Data Parsing
random.seed(42)
#Open and read the dataset file
with open("dataset_", "r") as f:
    lines = f.readlines()

#Skip to where the data actually starts in the file
data_start = next(i for i, line in enumerate(lines) if "@DATA" in line)

#From here parse the dataset into a usable dataset
dataset = pd.read_csv("dataset_", skiprows=data_start+1, header=None)

#Get the feature column
dataset.columns = ['id','rated','created_at','last_move_at','turns','victory_status','winner','increment_code','white_id','white_rating','black_id','black_rating','moves','opening_eco','opening_name','opening_ply']

#Look at the dataset before preprocessing
dataset.head()



FileNotFoundError: [Errno 2] No such file or directory: 'dataset_'

####Data Preprocessing

In [ ]:
# Additional Feature Extraction, Some additional features can be extracted to provide relevent data

#rating_diff: the difference in rating between players
#base_time: the base time amount set for the game
#time_increment: the time gained per move
#opening_strategy: Strategy of opening correlating to eco code letter (A, B, C, D, E)

dataset["rating_diff"] = dataset["white_rating"] - dataset["black_rating"]
dataset[["base_time", "time_increment"]] = dataset["increment_code"].str.split("+", expand=True)
dataset["opening_strategy"] = dataset["opening_eco"].str[0]


#Feature Refinement (Convert features to values that can be used in analysis)

#Convert boolean to integers
dataset["rated"] = dataset["rated"].astype(str).str.lower().map({"true": 1, "false": 0})

#Convert time to integers
dataset["base_time"] = dataset["base_time"].astype(int)
dataset["time_increment"] = dataset["time_increment"].astype(int)

#Get unique values of opening code and victory type
opening_strategy_values = dataset["opening_strategy"].unique()
winner_values = dataset["winner"].unique()
victory_status_values = dataset["victory_status"].unique()


#Data encoding (Encode non-numeric feature data into representative values that can be used for anlaysis)

#Create encoder to convert string data to binary array data
encoder = OneHotEncoder(categories=[opening_strategy_values, winner_values, victory_status_values], handle_unknown="ignore", sparse_output=False)

#Encode the data to unique values
encoded_Data = encoder.fit_transform(dataset[["opening_strategy", "winner", "victory_status"]])

#Create a dataframe from the encoded data to be appended to the dataset
encoded_dataframe = pd.DataFrame(encoded_Data, columns=encoder.get_feature_names_out(["opening_strategy", "winner", "victory_status"]), index = dataset.index)

#Append the encoded data to the dataset
dataset = pd.concat([dataset, encoded_dataframe], axis=1)



dataset.head()

### Expoloratory Data Analysis (EDA)

##### Class Distributions

In [ ]:
# get the number of each game outcome
winner_class_counts = dataset["winner"].value_counts()

# get the number of each victory type
type_winner_class_counts = dataset["victory_status"].value_counts()

# create two plots in one figure
fig, axes = plt.subplots(1, 2, figsize=(12,5))

# plot winner distribution
winner_class_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Class Distribution of Game Outcomes")
axes[0].set_xlabel("Game Outcome")
axes[0].set_ylabel("Number of Games")

# plot victory type distribution
type_winner_class_counts.plot(kind="bar", ax=axes[1])
axes[1].set_title("Class Distribution of Types of Game Outcome")
axes[1].set_xlabel("Game Outcome Type")
axes[1].set_ylabel("Number of Games")

plt.tight_layout()
plt.show()

##### Feature Correlations

In [ ]:
#use only the numeric features
numeric_features = dataset[["rated", "turns", "white_rating", "black_rating", "opening_ply", "rating_diff", "time_increment", "base_time"] + list(encoded_dataframe.columns)]

#Create a correlation matrix
corr_matrix = numeric_features.corr()

# Plot feature heatmap
plt.figure()
seaborn.heatmap(corr_matrix)

plt.title("Feature Correlation Heatmap")
plt.show()

### Training-Test Split

In [ ]:

#Create training data for Random Forest Model
rf_dataset = dataset.copy()

#Add desired features to be used -
features = ["rated", "turns", "white_rating", "black_rating", "opening_ply", "rating_diff", "base_time", "time_increment"]

#Add encoded opening strategy
opening_cols = [col for col in rf_dataset.columns if "opening_strategy_" in col]

#Create the total features to be used
x = rf_dataset[features + opening_cols]

#Define what we are trying to predict
y_winner = rf_dataset["winner"]
y_status = rf_dataset["victory_status"]

#Create the training and testing data
x_train, x_test, y_winner_train, y_winner_test, y_status_train, y_status_test = train_test_split( x, y_winner, y_status,
    test_size=0.2, random_state=42, stratify=y_winner)



## Random Forest

#### Hyperparameter tuning (through Grid Search Cross-Val) and model instantiation

In [ ]:
rf_estimators = 250

#Define estimators number here to allow for easy manipulation if necesary
param_distributions = {
    'n_estimators': [100, 200, 300, 400],           # Number of trees
    'max_depth': [None, 10, 20, 30, 40],            # Max depth of the trees
    'min_samples_split': [2, 5, 10],                # Minimum samples needed to split a node
    'min_samples_leaf': [1, 2, 4],                  # Minimum samples needed at a leaf node
    'max_features': ['sqrt', 'log2', None]          # Number of features to consider at every split
}

#Define Random Forest Classifier for determineing the Winner and the Victory Status
rf_winner_base = RandomForestClassifier(random_state=42, class_weight="balanced")
rf_status = RandomForestClassifier(n_estimators=rf_estimators, random_state=42, class_weight="balanced")


# 3. Set up the Randomized Search

winner_search = RandomizedSearchCV(
    estimator=rf_winner_base,
    param_distributions=param_distributions,
    n_iter=15,
    cv=5,
    scoring="f1_weighted",
    random_state=42,
    n_jobs=-1,
    verbose=1 # Prints progress to the console
)

print("\nOptimizing Winner Model...")
winner_search.fit(x_train, y_winner_train)
print("Best parameters for Winner Model:", winner_search.best_params_)


# Victory status model CV
# 5 Fold Split
status_f1_scores = cross_val_score(rf_status, x_train, y_status_train, cv=5, scoring="f1_weighted")

print("\nStatus weighted F1 scores:", status_f1_scores)
print("Status Mean weighted F1:", status_f1_scores.mean())

#Train models
rf_winner_best = winner_search.best_estimator_
rf_status.fit(x_train, y_status_train)


#test models
y_winner_pred = rf_winner_best.predict(x_test)
y_status_pred = rf_status.predict(x_test)


# Winner metrics
winner_accuracy = accuracy_score(y_winner_test, y_winner_pred)
winner_precision = precision_score(y_winner_test, y_winner_pred, average="weighted", zero_division=0)
winner_recall = recall_score(y_winner_test, y_winner_pred, average="weighted", zero_division=0)

# Status metrics
status_accuracy = accuracy_score(y_status_test, y_status_pred)
status_precision = precision_score(y_status_test, y_status_pred, average="weighted")
status_recall = recall_score(y_status_test, y_status_pred, average="weighted")

#Model Performance
print("\n--- Optimized Winner Model Performance ---")
print("Accuracy:", winner_accuracy)
print("Precision (weighted):", winner_precision)
print("Recall (weighted):", winner_recall)

print("\nVictory Status Model")
print("Accuracy:", status_accuracy)
print("Precision (weighted):", status_precision)
print("Recall (weighted):", status_recall)

## Transformer

#### Feature Engineering and Vectorization

In [ ]:
random.seed(42)
np.random.seed(42)

transformer_dataset = dataset.copy()

#Clean moves for transformer analysis
cleaned_moves = []

#Iterate through the moves and clean the moves removing quotes and unneeded white space
for move_string in transformer_dataset["moves"]:
    move_string = str(move_string).strip().strip("'")
    cleaned_moves.append(move_string)
transformer_dataset["moves_clean"] = cleaned_moves

#Encode winner and status label
winner_encoder = LabelEncoder()
status_encoder = LabelEncoder()

transformer_dataset["y_winner"] = winner_encoder.fit_transform(transformer_dataset["winner"])
transformer_dataset["y_status"] = status_encoder.fit_transform(transformer_dataset["victory_status"])

n_winner_classes = len(winner_encoder.classes_)
n_status_classes = len(status_encoder.classes_)

print("Winner classes :", list(winner_encoder.classes_))
print("Status classes :", list(status_encoder.classes_))

#Split data based on testing and training data, 80/20 split
train_df, test_df = train_test_split(
    transformer_dataset,
    test_size=0.2,
    random_state=42,
    stratify=transformer_dataset["y_winner"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

#Takes a move string and FEN and a progress fraction (0.3, 0.5, 0.9, etc.) and returns the progress of the game up to that percent
def safe_make_snapshot(move_string, progress):
    """
    Create truncated move string + FEN from a given progress fraction.
    progress should be something like 0.3, 0.5, 0.9
    """
    moves = str(move_string).split()

    if len(moves) < 5:
        board = chess.Board()
        return move_string, board.fen(), progress

    # cutoff based on chosen progress
    cutoff = max(1, int(progress * len(moves)))

    # avoid taking the very final plies to reduce leakage
    if len(moves) >= 8:
        cutoff = min(cutoff, len(moves) - 2)

    cutoff = max(1, cutoff)
    partial_moves = moves[:cutoff]

    board = chess.Board()

    try:
        for m in partial_moves:
            board.push_san(m)

        # If snapshot is already terminal, back up one move if possible
        if board.is_game_over() and len(partial_moves) > 1:
            board = chess.Board()
            partial_moves = partial_moves[:-1]
            for m in partial_moves:
                board.push_san(m)

        truncated_string = " ".join(partial_moves)
        fen = board.fen()

    except Exception:
        # fallback if SAN parsing fails
        print("Should not see this print")
        board = chess.Board()
        truncated_string = move_string
        fen = board.fen()

    return truncated_string, fen, progress

# Expand each game into multiple snapshots. Returns a new dataframe with one row per snapshot.
def expand_games(df, progress_list):

    expanded_rows = []

    for _, row in df.iterrows():
        row_dict = row.to_dict()

        for p in progress_list:
            truncated_moves, fen, progress = safe_make_snapshot(row_dict["moves_clean"], p)

            new_row = row_dict.copy()
            new_row["moves_truncated"] = truncated_moves
            new_row["fen"] = fen
            new_row["progress"] = progress
            expanded_rows.append(new_row)

    expanded_df = pd.DataFrame(expanded_rows).reset_index(drop=True)
    return expanded_df

#Convert the FEN board state to a vector
def fen_to_vector(fen):
    board = chess.Board(fen)

    vec = np.zeros(64 + 9, dtype=np.float32)

    for square, piece in board.piece_map().items():
        value = piece.piece_type
        if piece.color == chess.BLACK:
            value *= -1
        vec[square] = value

    # side to move
    vec[64] = 1.0 if board.turn == chess.WHITE else 0.0

    # castling rights
    vec[65] = 1.0 if board.has_kingside_castling_rights(chess.WHITE) else 0.0
    vec[66] = 1.0 if board.has_queenside_castling_rights(chess.WHITE) else 0.0
    vec[67] = 1.0 if board.has_kingside_castling_rights(chess.BLACK) else 0.0
    vec[68] = 1.0 if board.has_queenside_castling_rights(chess.BLACK) else 0.0

    # en passant available
    vec[69] = 1.0 if board.ep_square is not None else 0.0

    # halfmove clock
    vec[70] = float(board.halfmove_clock)

    # fullmove number
    vec[71] = float(board.fullmove_number)

    # in check
    vec[72] = 1.0 if board.is_check() else 0.0

    return vec

#These represent what percent of moves the model is being trained and tested on (eg first 30% of the game, first 60% of the game, first 90% of the game)
#Training the model on repeats of the same game cut to different length helps to allow the model to learn on early game, mid game, and late game strategies
#The model is tested on the first 95% of the moves because if the string contains the last move as checkmate it would effectively leak the label
train_progress_points = [0.2, 0.4, 0.6, 0.8, 0.95]
test_progress_points = [0.95]

train_expanded = expand_games(train_df, train_progress_points)
test_expanded = expand_games(test_df, test_progress_points)

print("\nOriginal training games:", len(train_df))
print("Expanded training snapshots:", len(train_expanded))
print("Original testing games:", len(test_df))
print("Expanded testing snapshots:", len(test_expanded))

#Subtokenization for the move tokens.
def san_to_subtokens(san):
    """
    Convert one SAN move like:
      Qxe4+: [Q, x, dst_e4, check]
      Nbd2: [N, src_b, dst_d2]
      cxd5: [P, src_c, x, dst_d5]
      O-O: [castle_kingside]
      O-O-O: [castle_queenside]
      e8=Q+: [P, dst_e8, promo_Q, check]
    """
    san = str(san).strip()

    # Remove common annotations if present
    san = san.replace("!", "").replace("?", "")

    # Castling
    if san == "O-O":
        return ["castle_kingside"]
    if san == "O-O-O":
        return ["castle_queenside"]

    tokens = []

    # Check / mate suffix
    suffix_tokens = []
    while len(san) > 0 and san[-1] in ["+", "#"]:
        if san[-1] == "+":
            suffix_tokens.append("check")
        elif san[-1] == "#":
            suffix_tokens.append("mate")
        san = san[:-1]

    # Promotion
    promo_match = re.search(r"=([QRBN])", san)
    if promo_match:
        promo_piece = promo_match.group(1)
        san = re.sub(r"=([QRBN])", "", san)
        promo_token = f"promo_{promo_piece}"
    else:
        promo_token = None

    # Destination square = final file-rank pair
    dest_match = re.search(r"([a-h][1-8])$", san)
    if not dest_match:
        return [f"raw_{san}"] + suffix_tokens

    dest = dest_match.group(1)
    san_core = san[:-2]

    # Piece type
    if len(san_core) > 0 and san_core[0] in "KQRBN":
        piece = san_core[0]
        san_core = san_core[1:]
    else:
        piece = "P"   # pawn

    tokens.append(piece)

    # Capture
    if "x" in san_core:
        before_capture, after_capture = san_core.split("x", 1)
        if before_capture:
            for ch in before_capture:
                tokens.append(f"src_{ch}")
        tokens.append("x")
        if after_capture:
            for ch in after_capture:
                tokens.append(f"hint_{ch}")
    else:
        if san_core:
            for ch in san_core:
                tokens.append(f"src_{ch}")

    tokens.append(f"dst_{dest}")

    if promo_token is not None:
        tokens.append(promo_token)

    tokens.extend(reversed(suffix_tokens))
    return tokens


def move_string_to_subtokens(move_string):
    """
    Convert a full move string into one flat token string with move boundaries.
    Example:
      "d4 d5 Nf3"
    becomes something like:
      "<m> P dst_d4 <m> P dst_d5 <m> N dst_f3"
    """
    moves = str(move_string).split()
    all_tokens = []

    for mv in moves:
        all_tokens.append("<m>")
        all_tokens.extend(san_to_subtokens(mv))

    return " ".join(all_tokens)


# Build subtokenized move strings from your truncated games
train_expanded["moves_subtok"] = train_expanded["moves_truncated"].apply(move_string_to_subtokens)
test_expanded["moves_subtok"] = test_expanded["moves_truncated"].apply(move_string_to_subtokens)

# Inspect examples
print("Original move string:")
print(train_expanded.loc[0, "moves_truncated"])

print("\nSubtokenized move string:")
print(train_expanded.loc[0, "moves_subtok"])

# Recompute sequence lengths on subtokenized strings
train_subtok_lengths = train_expanded["moves_subtok"].apply(lambda x: len(str(x).split()))
sequence_length = int(np.percentile(train_subtok_lengths, 95))
sequence_length = max(sequence_length, 20)

print("\nChosen subtoken sequence length:", sequence_length)

# New vectorizer on subtokens
vectorizer = TextVectorization(
    max_tokens=30000,
    output_mode="int",
    output_sequence_length=sequence_length,
    standardize=None,
    split="whitespace"
)

train_move_text = train_expanded["moves_subtok"].astype(str).values
test_move_text = test_expanded["moves_subtok"].astype(str).values

vectorizer.adapt(train_move_text)
vocab_size = len(vectorizer.get_vocabulary())

X_moves_train = vectorizer(train_move_text).numpy()
X_moves_test = vectorizer(test_move_text).numpy()

print("\nTrain move vector shape:", X_moves_train.shape)
print("Test move vector shape:", X_moves_test.shape)
print("Vocabulary size:", vocab_size)
print("\nFirst 50 vocabulary tokens:")
print(vectorizer.get_vocabulary()[:50])

#Numeric features to incorporate
numeric_cols = ["white_rating", "black_rating", "rating_diff", "base_time", "time_increment"]

# ECO letter
train_expanded["opening_letter"] = train_expanded["opening_eco"].astype(str).str[0]
test_expanded["opening_letter"] = test_expanded["opening_eco"].astype(str).str[0]

# Force consistent ECO columns across train/test
eco_categories = ["A", "B", "C", "D", "E"]

train_expanded["opening_letter"] = pd.Categorical(train_expanded["opening_letter"], categories=eco_categories)
test_expanded["opening_letter"] = pd.Categorical(test_expanded["opening_letter"], categories=eco_categories)

train_opening_onehot = pd.get_dummies(train_expanded["opening_letter"], prefix="eco")
test_opening_onehot = pd.get_dummies(test_expanded["opening_letter"], prefix="eco")

X_opening_train = train_opening_onehot.values.astype(np.float32)
X_opening_test = test_opening_onehot.values.astype(np.float32)

# Numeric features
X_numeric_train = train_expanded[numeric_cols + ["progress"]].astype(float).values
X_numeric_test = test_expanded[numeric_cols + ["progress"]].astype(float).values

# FEN features
X_fen_train = np.array([fen_to_vector(f) for f in train_expanded["fen"]], dtype=np.float32)
X_fen_test = np.array([fen_to_vector(f) for f in test_expanded["fen"]], dtype=np.float32)

# Scale numeric + fen on TRAIN ONLY
numeric_scaler = StandardScaler()
fen_scaler = StandardScaler()

X_numeric_train_scaled = numeric_scaler.fit_transform(X_numeric_train)
X_numeric_test_scaled = numeric_scaler.transform(X_numeric_test)

X_fen_train_scaled = fen_scaler.fit_transform(X_fen_train)
X_fen_test_scaled = fen_scaler.transform(X_fen_test)

# Final extra features
X_extra_train = np.hstack([
    X_numeric_train_scaled,
    X_opening_train,
    X_fen_train_scaled
]).astype(np.float32)

X_extra_test = np.hstack([
    X_numeric_test_scaled,
    X_opening_test,
    X_fen_test_scaled
]).astype(np.float32)

print("\nX_extra_train shape:", X_extra_train.shape)
print("X_extra_test shape:", X_extra_test.shape)


y_winner_train = train_expanded["y_winner"].values
y_winner_test = test_expanded["y_winner"].values

y_status_train = train_expanded["y_status"].values
y_status_test = test_expanded["y_status"].values


print("\nFirst training truncated move string:")
print(train_expanded.loc[0, "moves_truncated"])

print("\nFirst training FEN:")
print(train_expanded.loc[0, "fen"])

print("\nFirst training numeric (scaled):")
print(X_numeric_train_scaled[0])

print("\nFirst training opening one-hot:")
print(X_opening_train[0])

print("\nFirst training FEN vector (scaled):")
print(X_fen_train_scaled[0])

print("\nVocabulary size:", vocab_size)
print("Winner classes :", list(winner_encoder.classes_))
print("Status classes :", list(status_encoder.classes_))

#### Class Weight Balancing and Token/Positional Embedding

In [ ]:

winner_weights_auto = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_winner_train),
    y=y_winner_train
)

status_weights_auto = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_status_train),
    y=y_status_train
)

winner_class_weights = dict(enumerate(winner_weights_auto))
status_class_weights = dict(enumerate(status_weights_auto))

winner_class_weights = {k: float(np.sqrt(v)) for k, v in winner_class_weights.items()}
status_class_weights = {k: float(np.sqrt(v)) for k, v in status_class_weights.items()}


winner_sample_weights = np.array(
    [winner_class_weights[label] for label in y_winner_train],
    dtype=np.float32
)

status_sample_weights = np.array(
    [status_class_weights[label] for label in y_status_train],
    dtype=np.float32
)

print("Winner class weights:", winner_class_weights)
print("Status class weights:", status_class_weights)


early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-5
)

#Combines token embedding + learnable positional embedding.
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True)
        self.pos_emb = Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        tok = self.token_emb(x)
        return tok + positions

    def compute_mask(self, inputs, mask=None):
        return self.token_emb.compute_mask(inputs)

class MaskedGlobalAveragePooling1D(layers.Layer):
    def call(self, inputs, mask=None):
        if mask is None:
            return tf.reduce_mean(inputs, axis=1)

        mask = tf.cast(mask, inputs.dtype)
        mask = tf.expand_dims(mask, axis=-1)
        masked_inputs = inputs * mask
        sum_inputs = tf.reduce_sum(masked_inputs, axis=1)
        denom = tf.reduce_sum(mask, axis=1)
        return sum_inputs / tf.maximum(denom, tf.ones_like(denom))

    def compute_mask(self, inputs, mask=None):
        return None

#### Model Instantiation

In [ ]:
#One transformer encoder layer: Multi-Head Attention + FFN.
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)

        #Multihead Self attenton
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

        #Feed Forward Network
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])

        #layer Normalization
        self.ln1 = LayerNormalization(epsilon=1e-6)
        self.ln2 = LayerNormalization(epsilon=1e-6)

        #Dropout
        self.drop1 = Dropout(rate)
        self.drop2 = Dropout(rate)

    def call(self, x, training=False, mask=None):
        attention_mask = None

        if mask is not None:
            # shape: (batch, seq_len) -> (batch, 1, seq_len)
            attention_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)

        attn_out = self.att(
            x, x,
            attention_mask=attention_mask,
            training=training
        )
        attn_out = self.drop1(attn_out, training=training)
        out1 = self.ln1(x + attn_out)

        ffn_out = self.ffn(out1)
        ffn_out = self.drop2(ffn_out, training=training)
        return self.ln2(out1 + ffn_out)

    def compute_mask(self, inputs, mask=None):
        return mask

#Transformer that uses a hybrid of string and numeric features to predict both winner and status
def build_hybrid_transformer(vocab_size, maxlen, extra_dim, n_winner_classes, n_status_classes, embed_dim=64, num_heads=4, ff_dim=128):

    # Embeds the move tokens and positions and runs them through a transformer block
    moves_input = Input(shape=(maxlen,), name="moves_input")
    x_seq = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)(moves_input)
    x_seq = TransformerBlock(embed_dim, num_heads, ff_dim)(x_seq)
    x_seq = TransformerBlock(embed_dim, num_heads, ff_dim)(x_seq)
    x_seq = MaskedGlobalAveragePooling1D()(x_seq)
    x_seq = Dropout(0.1)(x_seq)

    # Extra numeric features are given as input
    extra_input = Input(shape=(extra_dim,), name="extra_input")
    x_extra = Dense(128, activation="relu")(extra_input)
    x_extra = Dropout(0.1)(x_extra)
    x_extra = Dense(64, activation="relu")(x_extra)
    x_extra = Dropout(0.1)(x_extra)

    # Combines the moves list and extra features into one representation of the game
    x = Concatenate()([x_seq, x_extra])
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.2)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.1)(x)

    # Winner head
    x_winner = Dense(64, activation="relu")(x)
    x_winner = Dropout(0.1)(x_winner)
    winner_output = Dense(n_winner_classes, activation="softmax", name="winner_output")(x_winner)

    # Status head
    x_status = Dense(64, activation="relu")(x)
    x_status = Dropout(0.1)(x_status)
    status_output = Dense(n_status_classes, activation="softmax", name="status_output")(x_status)

    #Define model inputs (moces and extras) and outputs (winner and status)
    model = Model(
        inputs=[moves_input, extra_input],
        outputs=[winner_output, status_output]
    )

    # Compile the model
    model.compile(
        optimizer="adam",
        loss={
            "winner_output": "sparse_categorical_crossentropy",
            "status_output": "sparse_categorical_crossentropy"
        },
        loss_weights={
            "winner_output": 1.0,
            "status_output": 1.3
        },
        metrics={
            "winner_output": ["accuracy"],
            "status_output": ["accuracy"]
        }
    )

    return model

# Build the model
hybrid_model = build_hybrid_transformer(
    vocab_size=vocab_size,
    maxlen=sequence_length,
    extra_dim=X_extra_train.shape[1],
    n_winner_classes=n_winner_classes,
    n_status_classes=n_status_classes
)

#Print the model summary
hybrid_model.summary()

#Train the Model
history = hybrid_model.fit(
    [X_moves_train, X_extra_train],
    [y_winner_train, y_status_train],
    sample_weight=[winner_sample_weights, status_sample_weights],
    validation_split=0.1,
    epochs=20,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Predict model on testing set
winner_pred_probs, status_pred_probs = hybrid_model.predict(
    {
        "moves_input": X_moves_test,
        "extra_input": X_extra_test
    },
    verbose=0
)

# Get the predicted labels
winner_pred = np.argmax(winner_pred_probs, axis=1)
status_pred = np.argmax(status_pred_probs, axis=1)

# Get the metrics of the model in regards to winner label
print("\n\n Winner Hybrid Transformer")
print(f"Accuracy: {accuracy_score(y_winner_test, winner_pred):.4f}")
print(f"Precision: {precision_score(y_winner_test, winner_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall: {recall_score(y_winner_test, winner_pred, average='weighted', zero_division=0):.4f}")
print(f"F1: {f1_score(y_winner_test, winner_pred, average='weighted', zero_division=0):.4f}")

# Get the metrics of the model in regards to status label
print("\n\n Victory Status Hybrid Transformer")
print(f"Accuracy: {accuracy_score(y_status_test, status_pred):.4f}")
print(f"Precision: {precision_score(y_status_test, status_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall: {recall_score(y_status_test, status_pred, average='weighted', zero_division=0):.4f}")
print(f"F1: {f1_score(y_status_test, status_pred, average='weighted', zero_division=0):.4f}")

### Tests Definition

In [ ]:
def build_phase_test_set(test_df, progress_value):
    phase_rows = []

    for _, row in test_df.iterrows():
        row_dict = row.to_dict()   # convert Series -> plain dict

        truncated_moves, fen, progress = safe_make_snapshot(row_dict["moves_clean"], progress_value)

        new_row = row_dict.copy()
        new_row["moves_truncated"] = truncated_moves
        new_row["fen"] = fen
        new_row["progress"] = progress
        phase_rows.append(new_row)

    phase_df = pd.DataFrame(phase_rows).reset_index(drop=True)

    # ECO letter
    eco_categories = ["A", "B", "C", "D", "E"]
    phase_df["opening_letter"] = phase_df["opening_eco"].astype(str).str[0]
    phase_df["opening_letter"] = pd.Categorical(phase_df["opening_letter"], categories=eco_categories)

    # Subtokenize moves to match training pipeline
    phase_df["moves_subtok"] = phase_df["moves_truncated"].apply(move_string_to_subtokens)

    # Move vectors
    phase_move_text = phase_df["moves_subtok"].astype(str).values
    X_moves_phase = vectorizer(phase_move_text).numpy()

    # Opening one-hot with fixed column order
    phase_opening_onehot = pd.get_dummies(phase_df["opening_letter"], prefix="eco")
    phase_opening_onehot = phase_opening_onehot.reindex(
        columns=[f"eco_{c}" for c in eco_categories],
        fill_value=0
    )
    X_opening_phase = phase_opening_onehot.values.astype(np.float32)

    # Numeric features
    X_numeric_phase = phase_df[numeric_cols + ["progress"]].astype(float).values
    X_numeric_phase_scaled = numeric_scaler.transform(X_numeric_phase)

    # FEN features
    X_fen_phase = np.array([fen_to_vector(f) for f in phase_df["fen"]], dtype=np.float32)
    X_fen_phase_scaled = fen_scaler.transform(X_fen_phase)

    # Final extra features
    X_extra_phase = np.hstack([
        X_numeric_phase_scaled,
        X_opening_phase,
        X_fen_phase_scaled
    ]).astype(np.float32)

    # Labels
    y_winner_phase = phase_df["y_winner"].values
    y_status_phase = phase_df["y_status"].values

    return phase_df, X_moves_phase, X_extra_phase, y_winner_phase, y_status_phase


#### Evaluation

In [ ]:
def evaluate_phase(progress_value):
    phase_df, X_moves_phase, X_extra_phase, y_winner_phase, y_status_phase = build_phase_test_set(
        test_df, progress_value
    )

    winner_pred_probs, status_pred_probs = hybrid_model.predict(
        [X_moves_phase, X_extra_phase],
        verbose=0
    )

    winner_pred = np.argmax(winner_pred_probs, axis=1)
    status_pred = np.argmax(status_pred_probs, axis=1)

    print(f"\n{'='*60}")
    print(f"TEST RESULTS AT {int(progress_value*100)}% OF GAME")
    print(f"{'='*60}")

    print("\nWinner Metrics")
    print(f"Accuracy : {accuracy_score(y_winner_phase, winner_pred):.4f}")
    print(f"Precision: {precision_score(y_winner_phase, winner_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_winner_phase, winner_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1       : {f1_score(y_winner_phase, winner_pred, average='weighted', zero_division=0):.4f}")

    print("\nVictory Status Metrics")
    print(f"Accuracy : {accuracy_score(y_status_phase, status_pred):.4f}")
    print(f"Precision: {precision_score(y_status_phase, status_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_status_phase, status_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1       : {f1_score(y_status_phase, status_pred, average='weighted', zero_division=0):.4f}")

    winner_cm = confusion_matrix(y_winner_phase, winner_pred)
    status_cm = confusion_matrix(y_status_phase, status_pred)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    winner_disp = ConfusionMatrixDisplay(
        confusion_matrix=winner_cm,
        display_labels=winner_encoder.classes_
    )
    winner_disp.plot(ax=axes[0], colorbar=False)
    axes[0].set_title(f"Winner Confusion Matrix ({int(progress_value*100)}%)")

    status_disp = ConfusionMatrixDisplay(
        confusion_matrix=status_cm,
        display_labels=status_encoder.classes_
    )
    status_disp.plot(ax=axes[1], colorbar=False)
    axes[1].set_title(f"Status Confusion Matrix ({int(progress_value*100)}%)")

    plt.tight_layout()
    plt.show()



for phase in [0.2, 0.4, 0.6, 0.8, 0.95]:
    evaluate_phase(phase)